# Setup


In [1]:
import os
import sys

import numpy as np
import pandas as pd
import psycopg2
from sqlalchemy import create_engine

from preprocessing import *

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from xgboost import XGBClassifier
from features import *

DB_URL = (
"postgresql+psycopg://neondb_owner:npg_Bo2SUY6ngypR@"
    "ep-orange-frost-afcl94sd-pooler.c-2.us-west-2.aws.neon.tech/"
    "neondb?sslmode=require"

)

engine = create_engine(
        DB_URL,
        pool_pre_ping=True,
        pool_recycle=3600
        )

In [2]:
#Loading Data
df = pd.read_sql(
    """
    SELECT *
    FROM ml.fight_dataset
    """,
    engine
)

## Preprocessing

In [3]:
# Preprocess
df = preprocess_ranks(df)
df = preprocess_weight(df)

df = df.reset_index(drop=True)
df["fight_id"] = df.index

history = build_fighter_history(df)

df = add_fighter_cumulative_features(df, history)
df = add_striking_rolling_features(df, history)
df = add_grappling_rolling_features(df, history)

In [4]:
df = create_features(df)

In [7]:
list(df.columns)

['fight_url',
 'event_url',
 'event_name',
 'event_date',
 'location_city',
 'location_state',
 'location_country',
 'referee',
 'weight_class',
 'gender',
 'title_fight',
 'num_rounds',
 'fighter_1',
 'fighter_2',
 'fighter_1_url',
 'fighter_2_url',
 'winner',
 'result',
 'result_details',
 'finish_round',
 'finish_time',
 'fighter_1_height_cm',
 'fighter_1_weight_lbs',
 'fighter_1_reach_cm',
 'fighter_1_stance',
 'fighter_1_dob',
 'fighter_2_height_cm',
 'fighter_2_weight_lbs',
 'fighter_2_reach_cm',
 'fighter_2_stance',
 'fighter_2_dob',
 'fighter_1_wins',
 'fighter_1_losses',
 'fighter_1_draws',
 'fighter_1_slpm',
 'fighter_1_str_acc',
 'fighter_1_sapm',
 'fighter_1_str_def',
 'fighter_1_td_avg',
 'fighter_1_td_acc',
 'fighter_1_td_def',
 'fighter_1_sub_avg',
 'fighter_2_wins',
 'fighter_2_losses',
 'fighter_2_draws',
 'fighter_2_slpm',
 'fighter_2_str_acc',
 'fighter_2_sapm',
 'fighter_2_str_def',
 'fighter_2_td_avg',
 'fighter_2_td_acc',
 'fighter_2_td_def',
 'fighter_2_sub_avg',

In [5]:
df["fighter_1_win"] = (df["winner"] == df["fighter_1"]).astype(int)
df["fighter_1_win"].value_counts()

fighter_1_win
1    5519
0    3239
Name: count, dtype: int64